# What is an Agent?

An agent is a helper that can do work on your behalf. Instead of just answering questions, it can take steps toward a goal, like looking up information, making decisions, or completing small tasks, while keeping track of what it is doing. You can think of it like a smart assistant that follows your instructions, uses available tools when needed, and tries to get a job done from start to finish.

The core idea is **thought -> action**:

- Thought: "I need to create a folder for these notes."
- Action: *calls a function that creates a folder*
- Thought: "Now I need to write a file inside it."
- Action: *calls a function that writes the file*
- Thought: "Task done, nothing left to do."

An LLM on its own only produces text. What turns it into an *agent* is wiring it up to **tools** (real Python functions with real side effects) and a **loop** that lets it decide, step by step, whether to call a tool or stop.

This notebook builds that intuition from the ground up, in LangChain:

1. The core components an agent needs (LLM, tools, loop).
2. A tiny agent built **by hand** — no framework magic, just `bind_tools()` and a `while` loop.
3. The same agent rebuilt with `create_agent()`, LangChain's higher-level abstraction — so you can see exactly what it's doing for you.

Notebook `1.0-langchain-fundamentals.ipynb` goes much deeper on every primitive touched here (messages, tools, memory, streaming, structured output). This notebook is just the conceptual on-ramp.

## The core components of an agent

Every agent, no matter how fancy the framework, is built from three pieces:

| Component | Role |
|---|---|
| **LLM** | The reasoning engine. Given a conversation, it decides what to say — or which tool to call. |
| **Tools** | Plain functions the LLM can ask to run (create a file, search the web, do math...). The LLM never executes them directly; *your code* does, then reports the result back. |
| **Loop** | The glue. Call the model -> did it ask for a tool? -> run the tool -> feed the result back -> call the model again -> repeat until it stops asking for tools (or you hit a safety limit). |

That's it. `create_agent()` later in this notebook is just a packaged version of that loop.

## Setup

Load environment variables and set up a sandbox folder so our toy file tools don't make a mess of the repo.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

SANDBOX_DIR = "./agent_workspace/00-intro-sandbox"
os.makedirs(SANDBOX_DIR, exist_ok=True)
print(f"Sandbox ready at: {os.path.abspath(SANDBOX_DIR)}")

## Just an LLM call

Before any tools or loops, remember: an LLM is "just" a function that turns messages into a message. `init_chat_model()` is LangChain's universal factory for building a chat model from a provider:model string.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

model = init_chat_model("openai:gpt-5.6-terra", reasoning_effort="none")

response = model.invoke([
    SystemMessage(content="You tell jokes."),
    HumanMessage(content="Tell me a short joke about a bald teacher explaining agents."),
])

print(response.content)

No tools, no loop — just a single request/response. `response` is an `AIMessage`. This is the "reasoning" half of an agent. Now let's give it something to *act* on.

## Defining tools with `@tool`

A tool is a plain Python function decorated with `@tool` from `langchain_core.tools`. LangChain reads the function's type hints and docstring to build the JSON schema the model sees — you never write that schema by hand.

We'll build two toy file-system tools our agent can use in a scratch folder, plus a tiny calculator for fun.

In [ ]:
import os
from langchain_core.tools import tool


@tool
def create_folder(folder_name: str) -> str:
    """Create a folder (directory) inside the sandbox workspace."""
    path = os.path.join(SANDBOX_DIR, folder_name)
    os.makedirs(path, exist_ok=True)
    return f"Folder created at: {path}"


@tool
def create_file(file_path: str, contents: str = "") -> str:
    """Create a file with optional text contents inside the sandbox workspace."""
    path = os.path.join(SANDBOX_DIR, file_path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(contents)
    return f"File created at: {path}"


@tool
def read_file(file_path: str) -> str:
    """Read and return the text contents of a file inside the sandbox workspace."""
    path = os.path.join(SANDBOX_DIR, file_path)
    with open(path, "r") as f:
        return f.read()


@tool
def calculator(a: float, b: float, operation: str) -> float:
    """Do basic arithmetic. operation must be one of '+', '-', '*', '/'."""
    if operation == "+":
        return a + b
    elif operation == "-":
        return a - b
    elif operation == "*":
        return a * b
    elif operation == "/":
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")


tools = [create_folder, create_file, read_file, calculator]
tools_by_name = {t.name: t for t in tools}
[t.name for t in tools]

## Building an agent from scratch

Now let's wire the LLM to those tools *manually*. `bind_tools()` attaches the tool schemas to the model, so the model can respond with `tool_calls` instead of (or alongside) plain text.

The recipe:

1. Call the model with `bind_tools()` on the current messages.
2. Look at `response.tool_calls` — a list of `{name, args, id}` dicts on the `AIMessage`.
3. For each tool call, run the matching Python function and wrap the result in a `ToolMessage` (matched back via `tool_call_id`).
4. Append everything to the message list and call the model again.
5. Stop when the model responds with no `tool_calls`.

First, a single manual step so you can see the shape of a tool call:

In [ ]:
model_with_tools = model.bind_tools(tools)

messages = [
    SystemMessage(content="You are a desktop assistant. Use tools when the user asks for file/folder actions."),
    HumanMessage(content="Create a folder called 'notes'."),
]

ai_message = model_with_tools.invoke(messages)
print("content:", repr(ai_message.content))
print("tool_calls:", ai_message.tool_calls)

The model didn't create anything itself — it just told us *which function to call and with what arguments*. Nothing has actually run on disk yet. That's our job:

In [ ]:
from langchain.messages import ToolMessage

messages.append(ai_message)

for call in ai_message.tool_calls:
    tool_fn = tools_by_name[call["name"]]
    result = tool_fn.invoke(call["args"])
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

for m in messages:
    m.pretty_print()

That's one round trip. A real task usually needs *several* rounds — the model might create a folder, then decide it also needs to create a file inside it, then stop. Let's wrap the pattern above into a loop that keeps going until the model has no more tool calls to make (or we hit a safety cap):

In [ ]:
def run_agent_loop(user_task: str, max_iters: int = 5, verbose: bool = True):
    """A minimal from-scratch agent loop on top of a LangChain chat model."""
    messages = [
        SystemMessage(content=(
            "You are a desktop file-system assistant. "
            "Use the available tools to satisfy the user's request. "
            "When the task is complete, reply with a short summary and stop calling tools."
        )),
        HumanMessage(content=user_task),
    ]

    for i in range(max_iters):
        ai_message = model_with_tools.invoke(messages)
        messages.append(ai_message)

        if not ai_message.tool_calls:
            if verbose:
                print(f"--- iter {i + 1}: no tool calls, agent is done ---")
            return ai_message.content

        if verbose:
            print(f"--- iter {i + 1}: {len(ai_message.tool_calls)} tool call(s) ---")

        for call in ai_message.tool_calls:
            tool_fn = tools_by_name[call["name"]]
            result = tool_fn.invoke(call["args"])
            if verbose:
                print(f"  -> {call['name']}({call['args']}) = {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    print(f"Reached max iterations ({max_iters}) — stopping.")
    return messages[-1].content


final_answer = run_agent_loop(
    "Create a folder called 'trip-notes'. Inside that folder, create a file called "
    "'day-1.md' with the contents 'Arrived in Lisbon!'. Then read the file back to confirm."
)
print("\nFinal answer:", final_answer)

That's a real, if tiny, agent: an LLM, a set of tools, and a loop that keeps calling the model and executing tools until the model decides it's done. Everything about LangChain's tool-calling primitives (`@tool`, `bind_tools()`, `ToolMessage`) is framework-native — the *loop itself* is the only part we wrote by hand.

## The same thing with `create_agent()`

`langchain.agents.create_agent()` packages exactly the loop we just wrote:

- It calls the model with tools bound.
- It inspects `tool_calls` on the returned `AIMessage`.
- It executes the matching tools and appends `ToolMessage`s.
- It repeats until the model stops calling tools.
- It manages the message list for you (as `state["messages"]`).

Let's solve the exact same toy task in about three lines.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,  # create_agent binds the tools for you — no manual bind_tools() needed
    tools=tools,
    system_prompt=(
        "You are a desktop file-system assistant. "
        "Use the available tools to satisfy the user's request."
    ),
)

result = agent.invoke({
    "messages": (
        "Create a folder called 'trip-notes-v2'. Inside that folder, create a file called "
        "'day-1.md' with the contents 'Arrived in Lisbon!'. Then read the file back to confirm."
    )
})

result["messages"][-1].pretty_print()

### What `create_agent()` replaced

| From-scratch piece | `create_agent()` equivalent |
|---|---|
| `model.bind_tools(tools)` | done internally when you pass `tools=[...]` |
| Manually appending `SystemMessage`/`HumanMessage` | `system_prompt=...` and `{"messages": "..."}` |
| Reading `ai_message.tool_calls` and dispatching to `tools_by_name` | handled internally |
| Building `ToolMessage(content=..., tool_call_id=...)` | handled internally |
| The `while`/`for` loop and `max_iters` safety cap | built into the compiled graph |
| Returning the final text | `result["messages"][-1]` |

Same behavior, three lines instead of thirty. Now that you've seen what's under the hood, using `create_agent()` should feel like a shortcut rather than a black box.

## What's next

Notebook `1.0-langchain-fundamentals.ipynb` goes deeper on every primitive touched here: messages, `init_chat_model()` across providers, tool design (structured args, dependency injection), memory/checkpointing, and streaming. This notebook was just about building the intuition for *why* those primitives exist — because you now know exactly what they replace.